<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Preguntarle a Cien Amigos: Random Forests 🌲
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 02
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/02%20-%20Arboles%20Bosques%20y%20Redes%20Neuronales/Para%20Dummies/01_Random_Forests_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del cuaderno 01 (Random Forests). Cuando tienes una decisión difícil, ¿le preguntas a **una sola persona** o a **varias**? Casi siempre la opinión **combinada** de muchos es más confiable que la de uno solo. Un **bosque aleatorio** es justo eso: **cientos de árboles de decisión votando**.

Al terminar podrás explicar:
1. Por qué **promediar muchas opiniones** reduce los errores (la «sabiduría de la multitud»).
2. Por qué cada árbol del bosque ve **una versión distinta** de los datos.
3. Cómo el bosque **se autoevalúa** con los datos que cada árbol no vio.
4. Qué significa que una variable sea «importante» (¡y por qué hay que tener cuidado!).

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../01_Random_Forests.ipynb). Otras miradas sencillas: [Random Forests (Dummies) en Data Mining](../../../Data%20Mining/04%20-%20Arboles%20de%20Decision%20y%20Bosques%20Aleatorios/Para%20Dummies/04_Random_Forests_Dummies.ipynb) y [Bagging y Random Forests (Dummies) en Data Science Programming](../../../Data%20Science%20programming/09%20-%20Decision%20Trees/Para%20Dummies/03_Metodos_Ensamble_Bagging_y_Random_Forests_Dummies.ipynb).

---
## 1. La sabiduría de la multitud 🫘

En una feria, se pide a 100 personas **adivinar cuántos frijoles hay en un frasco**. Casi nadie acierta: unos dicen 800, otros 1500. Pero si sacas el **promedio** de todas las respuestas, ¡suele quedar muy cerca de la cifra real! Los errores de **exceso** y de **defecto** se cancelan entre sí.

Se cuenta que algo parecido pasó en 1906, cuando el estadístico Francis Galton comparó el promedio de las estimaciones de cientos de visitantes de una feria de ganado sobre el peso de un buey con el peso verdadero: el promedio quedó muy cerca. Comprobémoslo con una **simulación**: 5 000 ferias, cada una con 100 personas que se equivocan al azar.

In [ ]:
# Simulacion: 5000 ferias; en cada una, 100 personas adivinan el numero de frijoles (real = 1000) y se equivocan al azar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
real = 1000
ferias, personas = 5000, 100
respuestas = real + rng.normal(0, 250, size=(ferias, personas))          # cada persona se equivoca ~250 frijoles (al azar)

error_una_persona = np.abs(respuestas[:, 0] - real)                      # lo que se equivoca "una sola persona"
error_promedio = np.abs(respuestas.mean(axis=1) - real)                  # lo que se equivoca el PROMEDIO de las 100

print(f"Error típico de UNA persona : {error_una_persona.mean():6.0f} frijoles")
print(f"Error típico del PROMEDIO   : {error_promedio.mean():6.0f} frijoles")

plt.figure(figsize=(9, 4.8))
plt.hist(respuestas[:, 0], bins=50, alpha=0.6, color="#f59e0b", label="Lo que dice UNA persona")
plt.hist(respuestas.mean(axis=1), bins=50, alpha=0.8, color="#16a34a", label="El PROMEDIO de 100 personas")
plt.axvline(real, color="black", linestyle="--", label="Valor real")
plt.xlabel("Número de frijoles estimado"); plt.ylabel("Número de ferias")
plt.title("El promedio de muchos se equivoca mucho menos", fontweight="bold")
plt.legend()
plt.show()

---
## 2. Cada amigo ve una parte distinta 🎒

Para que la multitud funcione, los amigos **no pueden ser copias**: si todos opinaran exactamente lo mismo, promediar no sirve de nada. Por eso, en un **bosque aleatorio** a cada árbol se le da:

- **Una muestra distinta de los datos**: como si a cada amigo le prestaras **un cuaderno con algunas hojas repetidas y otras faltantes** (se sortean con repetición).
- **Solo algunas columnas** en cada pregunta: como si, antes de cada pregunta, le taparas al amigo **parte de la información** para que no todos se fijen siempre en lo mismo.

Cada árbol queda **un poco distinto y un poco ciego**, pero al **votar todos juntos**, los errores individuales se compensan. Probémoslo con el conjunto de datos de **tumores (cáncer de mama)**: ¿un solo árbol o un bosque de 200?

In [ ]:
# Un arbol solo vs. un bosque de 200 arboles, evaluados con pacientes que NO vieron al entrenar
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

datos = load_breast_cancer()
X_ent, X_exa, y_ent, y_exa = train_test_split(datos.data, datos.target, test_size=0.3, stratify=datos.target, random_state=42)

un_arbol = DecisionTreeClassifier(random_state=42).fit(X_ent, y_ent)
bosque = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_ent, y_ent)

print(f"Un solo árbol : acierta {un_arbol.score(X_exa, y_exa):.1%} en el examen")
print(f"Bosque (200)  : acierta {bosque.score(X_exa, y_exa):.1%} en el examen")

# Repetimos con 20 repartos distintos de los datos para no depender de la suerte de uno solo
rng = np.random.default_rng(0)
dif = []
for r in range(20):
    a, b, c, d = train_test_split(datos.data, datos.target, test_size=0.3, stratify=datos.target, random_state=int(rng.integers(1000)))
    dif.append(RandomForestClassifier(n_estimators=100, random_state=r).fit(a, c).score(b, d) - DecisionTreeClassifier(random_state=r).fit(a, c).score(b, d))
print(f"\nEn 20 repartos distintos de los datos, el bosque ganó al árbol solo en {np.mean(np.array(dif) > 0):.0%} de las veces "
      f"(ventaja media: {np.mean(dif) * 100:.1f} puntos porcentuales).")

---
## 3. El bosque se autoevalúa 📝

Como cada árbol **no vio** más o menos **un tercio de los pacientes** (las «hojas del cuaderno» que le faltaron), esos pacientes sirven como **examen sorpresa** para ese árbol. Si para cada paciente consultamos **solo a los árboles que no lo vieron**, obtenemos una **nota honesta del bosque sin tener que apartar datos aparte**. Se llama **evaluación *out-of-bag*** («fuera de la bolsa»).

Es como estudiar con un grupo de amigos y que cada uno responda **solo las preguntas del examen que él no practicó**.

In [ ]:
# Nota "honesta" del bosque con los pacientes que cada arbol no vio (out-of-bag), sin usar el grupo de examen
bosque_oob = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=42).fit(X_ent, y_ent)
print(f"Nota de autoevaluación del bosque (solo con datos de entrenamiento): {bosque_oob.oob_score_:.1%}")
print(f"Nota en el examen real (pacientes apartados)                      : {bosque_oob.score(X_exa, y_exa):.1%}")
print("\nSon parecidas: el bosque puede estimar qué tan bien funcionará con datos nuevos usando solo lo que ya tiene.")

---
## 4. ¿Qué variables importan? 🔍

El bosque también puede decirte **qué datos usó más** para decidir (por ejemplo, el tamaño o la textura del tumor). Es muy útil para **entender el problema**. Pero ojo con dos advertencias:

- Que una variable sea «importante» **no significa que cause** el resultado: solo que el modelo la **usó mucho**.
- Si dos variables son **casi lo mismo** (como el radio y el perímetro de un tumor), el bosque **reparte** el crédito entre ellas: cada una parece menos importante de lo que es.

In [ ]:
# Las 8 variables que mas uso el bosque
imp = pd.Series(bosque_oob.feature_importances_, index=datos.feature_names).sort_values().tail(8)
plt.figure(figsize=(9, 4.8))
plt.barh(imp.index, imp.values, color="#f59e0b")
plt.xlabel("Importancia (cuánto usó el bosque esta variable)")
plt.title("Las variables que más usó el bosque para decidir", fontweight="bold")
plt.tight_layout()
plt.show()
print("Recuerda: 'importante' = 'muy usada por el modelo', no necesariamente 'causa del resultado'.")

---
##### 🛠️ Práctica 1: ¿Cuántos amigos hay que invitar al comité?

En la Sección 1 viste que promediar muchas opiniones ayuda. ¿Y cuántos árboles necesita el bosque? Averígualo con los datos de tumores (`X_ent`, `y_ent`, `X_exa`, `y_exa`) y el `bosque_oob` que ya entrenaste:

1. Crea un diccionario vacío `notas = {}`.
2. Recorre `[1, 5, 25, 100, 300]` (el número de árboles). Para cada `n`, entrena un `RandomForestClassifier(n_estimators=n, random_state=42)` con `X_ent`, `y_ent` y guarda en `notas[n]` cuánto acierta en el examen.
3. Convierte `notas` en una tabla con `pd.Series(notas)` e imprímela: ¿mejora siempre al agregar árboles?
4. De paso, averigua **cuál es la variable estrella** del `bosque_oob`: la de mayor `feature_importances_` (usa `argmax()` y `datos.feature_names`) y guárdala en `variable_estrella`.
5. Comprueba que el bosque de 300 árboles acierta al menos tanto como el de 1 solo árbol.

In [ ]:
# Escribe tu código aquí

# 1) El diccionario donde guardaremos las notas
# notas = {}

# 2) Probamos comites de distinto tamano
# for n in [1, 5, 25, 100, 300]:
#     comite = RandomForestClassifier(n_estimators=..., random_state=42).fit(X_ent, y_ent)
#     notas[n] = comite.score(X_exa, y_exa)

# 3) Mostramos la tabla
# print(pd.Series(notas).round(3))

# 4) La variable estrella
# variable_estrella = datos.feature_names[bosque_oob.feature_importances_.argmax()]
# print(variable_estrella)


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1) El diccionario donde guardaremos las notas
notas = {}

# 2) Probamos comites de distinto tamano
for n in [1, 5, 25, 100, 300]:
    comite = RandomForestClassifier(n_estimators=n, random_state=42).fit(X_ent, y_ent)
    notas[n] = comite.score(X_exa, y_exa)

# 3) Mostramos la tabla
print("Aciertos en el examen según el número de árboles:")
print(pd.Series(notas, name="aciertos").rename_axis("árboles").round(3).to_string())

# 4) La variable estrella
variable_estrella = datos.feature_names[bosque_oob.feature_importances_.argmax()]
print(f"\nLa variable que más usó el bosque: {variable_estrella}")

# 5) Comprobacion
assert notas[300] >= notas[1]
print("\nUn comité grande acierta al menos tanto como un solo árbol: más amigos opinando suele ayudar, aunque la mejora se aplana.")
```
</details>

---


---
## Resumen en una frase 🎯

> **Un bosque aleatorio es un comité de muchos árboles, cada uno con una versión distinta de los datos, cuyo voto combinado se equivoca menos que cualquiera de ellos; además se autoevalúa con los datos que cada árbol no vio y señala qué variables usó más.**

### Lo que aprendiste hoy:
- ✅ **Promediar muchas opiniones** (la sabiduría de la multitud) reduce los errores.
- ✅ Cada árbol ve **datos y columnas distintos** para que los amigos no sean copias.
- ✅ El bosque puede **autoevaluarse** (*out-of-bag*) con lo que cada árbol no vio.
- ✅ La **importancia** dice qué usó el modelo, no qué *causa* el resultado.

> 🎓 **Siguiente paso:** [Redes Neuronales (Dummies)](02_Redes_Neuronales_Dummies.ipynb). Versión con más detalle de este tema: [cuaderno estándar](../01_Random_Forests.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué promediar las opiniones de 100 personas ayuda **solo si no opinan todas exactamente igual**?
2. Piensa en una decisión importante de tu vida (comprar una casa, elegir una carrera). ¿A cuántas personas distintas consultarías y qué cuidado tendrías para que sus opiniones sean realmente **independientes**?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>